In [ ]:
!pip install -q openai gradio

In [ ]:
from openai import OpenAI
import gradio as gr

client = OpenAI(
    api_key="sk--jVShv6wQzUKSZRakUWd_Q",
    base_url="https://apidev.navigatelabsai.com"
)

In [ ]:
!pip install fpdf2

In [ ]:
def generate_study_plan(subject, topics, days_left, study_hours, level):

    prompt = f"""
You are an expert AI Study Planner.

Student Details

Subject: {subject}

Topics: {topics}

Exam in: {days_left} days

Study Hours Per Day: {study_hours}

Learning Level: {level}

Generate the response EXACTLY in this format.

### STUDY_PLAN
(Create a day-wise timetable.)

### EXPLANATION
(Explain every topic simply.)

### REVISION_NOTES
(Give concise revision notes.)

### QUIZ
(Create 5 multiple choice questions with answers.)

### EXAM_TIPS
(Give 5 useful exam preparation tips.)

Do not miss any section.
"""

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are an expert educational AI assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7
    )

    return response.choices[0].message.content


def split_response(text):

    sections = {
        "study_plan": "",
        "explanation": "",
        "revision": "",
        "quiz": "",
        "tips": ""
    }

    current = None

    for line in text.split("\n"):

        if "### STUDY_PLAN" in line:
            current = "study_plan"
            continue

        elif "### EXPLANATION" in line:
            current = "explanation"
            continue

        elif "### REVISION_NOTES" in line:
            current = "revision"
            continue

        elif "### QUIZ" in line:
            current = "quiz"
            continue

        elif "### EXAM_TIPS" in line:
            current = "tips"
            continue

        if current:
            sections[current] += line + "\n"

    return (
        sections["study_plan"],
        sections["explanation"],
        sections["revision"],
        sections["quiz"],
        sections["tips"],
        text
    )


def ai_study_assistant(subject, topics, days_left, study_hours, level):

    response = generate_study_plan(
        subject,
        topics,
        days_left,
        study_hours,
        level
    )

    return split_response(response)
def solve_doubt(subject, question):

    prompt = f"""
You are an expert teacher.

Subject: {subject}

Student Question:
{question}

Explain the answer in simple language.

Also provide:
1. A simple explanation
2. A real-world example
3. A short exam tip

Keep the answer easy to understand.
"""

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI tutor."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.5
    )

    return response.choices[0].message.content


def generate_quiz(subject, topic, difficulty):

    prompt = f"""
You are an expert teacher.

Create a quiz.

Subject: {subject}

Topic: {topic}

Difficulty: {difficulty}

Instructions:

1. Generate exactly 5 multiple-choice questions.
2. Each question must have four options (A, B, C, D).
3. DO NOT reveal the correct answers.
4. Format the quiz neatly.
5. At the end write:

"Attempt all the questions first. Then click the 'Show Answer Key' button to check your answers."

Return only the quiz.
"""

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are an AI Quiz Generator."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.6
    )

    return response.choices[0].message.content


def generate_answer_key(subject, topic, difficulty):

    prompt = f"""
You are an expert teacher.

Subject: {subject}

Topic: {topic}

Difficulty: {difficulty}

Generate ONLY the answer key for the quiz.

Format:

Answer Key

1. B
2. D
3. A
4. C
5. B

Also give a one-line explanation for each answer.

Do not generate the questions again.
"""

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are an AI Quiz Evaluator."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.5
    )

    return response.choices[0].message.content


# ===============================
# PDF DOWNLOAD FUNCTION
# ===============================

from fpdf import FPDF


def download_pdf(full_text):

    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    pdf.set_font("Arial", "B", 16)
    pdf.cell(0, 10, "StudyGenie AI Report", ln=True, align="C")

    pdf.ln(5)

    pdf.set_font("Arial", "", 12)

    # Remove emojis (FPDF doesn't support them)
    for emoji in ["🎓", "📅", "📖", "📝", "💡", "✅", "❓", "🚀"]:
        full_text = full_text.replace(emoji, "")

    # Convert unsupported characters safely
    full_text = full_text.encode("latin-1", "replace").decode("latin-1")

    pdf.multi_cell(0, 8, full_text)

    filename = "StudyGenie_Report.pdf"
    pdf.output(filename)

    return filename

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft(), title="StudyGenie AI") as demo:

    gr.Markdown("""
# 🎓 StudyGenie AI
### Personalized Study Planner using Generative AI

Generate a personalized study schedule, topic explanations, revision notes,
practice quizzes, and exam tips using AI.

---
### ✨ Features

✅ Personalized Study Plan

✅ AI Doubt Solver

✅ AI Quiz Generator

---
""")

    # -------------------------------
    # Study Planner Inputs
    # -------------------------------

    subject = gr.Textbox(
        label="📘 Subject",
        placeholder="Example: Python Programming"
    )

    topics = gr.Textbox(
        label="📚 Topics",
        lines=4,
        placeholder="Functions, Lists, Dictionaries, OOP..."
    )

    with gr.Row():

        days_left = gr.Slider(
            minimum=1,
            maximum=30,
            value=5,
            step=1,
            label="📅 Days Until Exam"
        )

        study_hours = gr.Slider(
            minimum=1,
            maximum=10,
            value=3,
            step=1,
            label="⏰ Study Hours Per Day"
        )

    level = gr.Dropdown(
        choices=["Beginner", "Intermediate", "Advanced"],
        value="Beginner",
        label="🎯 Learning Level"
    )

    with gr.Row():
     generate_btn = gr.Button(
        "🚀 Generate Study Plan",
        variant="primary"
    )

    pdf_btn = gr.Button(
        "📄 Generate PDF"
    )

    # -------------------------------
    # Output Tabs
    # -------------------------------

    with gr.Tabs():

        with gr.Tab("📅 Study Plan"):
            study_output = gr.Markdown()

            pdf_file = gr.File(
               label="📥 Download Study Plan PDF"
            )

        with gr.Tab("📖 Topic Explanation"):
            explanation_output = gr.Markdown()

        with gr.Tab("📝 Revision Notes"):
            revision_output = gr.Markdown()

        with gr.Tab("❓ Practice Quiz"):
            quiz_output = gr.Markdown()

        with gr.Tab("💡 Exam Tips"):
            tips_output = gr.Markdown()

        # =============================
        # AI DOUBT SOLVER
        # =============================

        with gr.Tab("❓ AI Doubt Solver"):

            doubt_subject = gr.Textbox(
                label="📘 Subject",
                placeholder="Example: Python Programming"
            )

            doubt_question = gr.Textbox(
                label="❓ Ask Your Doubt",
                lines=4,
                placeholder="Example: What is Recursion?"
            )

            doubt_btn = gr.Button(
                "🤖 Solve My Doubt",
                variant="primary"
            )

            doubt_output = gr.Markdown()

        # =============================
        # AI QUIZ GENERATOR
        # =============================

        with gr.Tab("📝 AI Quiz Generator"):

            quiz_subject = gr.Textbox(
                label="📘 Subject",
                placeholder="Example: Python Programming"
            )

            quiz_topic = gr.Textbox(
                label="📚 Topic",
                placeholder="Example: Functions"
            )

            quiz_difficulty = gr.Dropdown(
                choices=["Easy", "Medium", "Hard"],
                value="Medium",
                label="🎯 Difficulty"
            )

            with gr.Row():

                quiz_btn = gr.Button(
                    "📝 Generate Quiz",
                    variant="primary"
                )

                answer_btn = gr.Button(
                    "✅ Show Answer Key"
                )

            quiz_result = gr.Markdown()

            answer_result = gr.Markdown()
            hidden_text = gr.Textbox(
               visible=False
            )

    # -------------------------------
    # Button Connections
    # -------------------------------

    generate_btn.click(
        fn=ai_study_assistant,
        inputs=[
            subject,
            topics,
            days_left,
            study_hours,
            level
        ],
        outputs=[
            study_output,
            explanation_output,
            revision_output,
            quiz_output,
            tips_output,
            hidden_text
        ]
    )
    pdf_btn.click(
        fn=download_pdf,
        inputs=hidden_text,
        outputs=pdf_file
    )

    doubt_btn.click(
        fn=solve_doubt,
        inputs=[
            doubt_subject,
            doubt_question
        ],
        outputs=doubt_output
    )

    quiz_btn.click(
        fn=generate_quiz,
        inputs=[
            quiz_subject,
            quiz_topic,
            quiz_difficulty
        ],
        outputs=quiz_result
    )

    answer_btn.click(
        fn=generate_answer_key,
        inputs=[
            quiz_subject,
            quiz_topic,
            quiz_difficulty
        ],
        outputs=answer_result
    )

demo.launch(share=True)

/tmp/ipykernel_3442/3291354527.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="StudyGenie AI") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88b55a41e2a071511f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 19.8 MB/s eta 0:00:00


In [ ]:
from fpdf import FPDF

def create_pdf(study_plan):

    pdf = FPDF()
    pdf.add_page()

    pdf.set_font("Arial", "B", 16)
    pdf.cell(0, 10, "StudyGenie AI", ln=True, align="C")

    pdf.set_font("Arial", "", 12)
    pdf.multi_cell(0, 8, study_plan)

    filename = "StudyGenie_StudyPlan.pdf"
    pdf.output(filename)

    return filename

In [ ]:
print(generate_quiz("Python", "Functions", "Easy"))

Python Functions Quiz

1. What is the primary purpose of a function in Python?
A) To store data  
B) To perform a specific task or calculation  
C) To display information on the screen  
D) To create a new variable  

2. Which keyword is used to define a function in Python?
A) function  
B) def  
C) define  
D) func  

3. What does the following code do?
```python
def greet():
    print("Hello, World!")
greet()
```
A) Defines a variable named greet  
B) Prints "Hello, World!" to the screen  
C) Creates an error because of missing arguments  
D) Nothing, it will not run  

4. How do you call a function named `calculate_sum` in Python?
A) call calculate_sum()  
B) execute calculate_sum  
C) calculate_sum()  
D) run calculate_sum  

5. Which of the following is true about functions in Python?
A) Functions cannot return values  
B) Functions are only used for mathematical calculations  
C) Functions help organize code and can return results  
D) Functions are not reusable once defined  

A